# 08 — GPU/CPU training throughput check

**Decision this feeds**: before running the 5 rung-4 training experiments
(family-oversampling, LR schedule, augmentation, `class_weight`,
denoising — `README.md` Next steps), confirm the training loop is
actually GPU-bound (not stalled on CPU-side data loading) and measure
real per-fold wall-clock time, so we know whether 5 more rung-3-scale
runs (25 CNN trainings each) fit in the time remaining before
2026-09-16.

**No modeling decision is made here** — this notebook only measures, it
doesn't train a model that gets evaluated or gated. Cells 2-3 use
synthetic random tensors (no real data at all); if cell 1 shows no CUDA
device, stop here — the numbers below assume GPU training throughout
this project and won't be meaningful on CPU.

**Data handling**: cell 4 loads real `.nii.gz` volumes (via the shared
cache already validated in rungs 2/3), so per the AI-assistant data rule
(`README.md`) this notebook is **[RUN ME]** — run it yourself, share
back only the printed timing numbers, never any per-row output. Cells
1-3 touch no data at all and are included as `[RUN ME]` only for
consistency of numbering — nothing stops you from running them, there's
just nothing for me to read either way.

In [1]:
# [RUN ME] -- no data access. Device/environment check only.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total memory: {props.total_memory / 1e9:.1f} GB")
else:
    print("No CUDA device visible -- STOP HERE. config.DEVICE='cuda' is hardcoded "
          "throughout this project's training code; the throughput numbers below "
          "assume GPU and won't tell us anything useful on CPU.")

CUDA available: True
Device: NVIDIA GeForce RTX 4060 Laptop GPU
Total memory: 8.6 GB


In [2]:
# [RUN ME] -- no real data, synthetic random tensors only. Measures the
# model's raw GPU compute cost, decoupled from data loading, so cell 4
# below can tell us whether real-data throughput is GPU-bound or
# CPU-bound by comparison.
import time

import config
import model

device_type = "cuda" if str(config.DEVICE).startswith("cuda") else "cpu"

net = model.build_model().to(config.DEVICE)
optimizer = torch.optim.Adam(net.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY)
loss_fn = torch.nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler(device_type, enabled=config.USE_AMP)

x = torch.randn(config.BATCH_SIZE, 1, *config.TARGET_SHAPE, device=config.DEVICE)
y = torch.randint(0, 2, (config.BATCH_SIZE,), device=config.DEVICE, dtype=torch.float32)


def run_batches(n):
    for _ in range(n):
        optimizer.zero_grad()
        with torch.autocast(device_type=device_type, enabled=config.USE_AMP):
            loss = loss_fn(net(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()


run_batches(5)  # warmup -- excludes CUDA kernel compilation/allocator setup from the timing
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

N = 50
start = time.time()
run_batches(N)
torch.cuda.synchronize()
elapsed = time.time() - start

batches_per_sec = N / elapsed
gpu_only_volumes_per_sec = batches_per_sec * config.BATCH_SIZE
print(f"GPU-only throughput (synthetic data, batch_size={config.BATCH_SIZE}, AMP={config.USE_AMP}): "
      f"{batches_per_sec:.1f} batches/s ({gpu_only_volumes_per_sec:.0f} volumes/s)")
print(f"Peak GPU memory this run: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB total")

GPU-only throughput (synthetic data, batch_size=8, AMP=True): 132.0 batches/s (1056 volumes/s)
Peak GPU memory this run: 0.10 GB / 8.6 GB total


In [3]:
# [RUN ME] -- loads real pixel data + row-level labels. Times cache
# build/reuse and one real fold's worth of training, and compares
# real-data throughput against the previous cell's GPU-only ceiling to
# see whether data loading is the bottleneck. num_workers=0 -- see
# notebook 06's cell 2 for why.
import numpy as np
import pandas as pd

import cache
import dataset
import evaluate

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache",
    config_fingerprint=config_fingerprint,
)
cache_elapsed = time.time() - cache_start
print(f"cache {'REUSED' if volume_cache.was_reused else 'REBUILT'} in {cache_elapsed:.1f}s "
      f"for {len(uids)} volumes"
      + ("" if volume_cache.was_reused
         else f" ({cache_elapsed / len(uids) * 1000:.1f} ms/volume)"))

# Time one real fold's worth of forward+backward passes (cache is warm
# now, so this isolates DataLoader/collate overhead, not preprocessing).
outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                   n_splits=config.N_FOLDS, random_state=config.SEED)
train_idx, _ = outer_folds[0]
fold_uids = [uids[i] for i in train_idx]
fold_labels = [labels[i] for i in train_idx]
fold_ds = dataset.DatParkinsonDataset(fold_uids, fold_labels, load_fn=volume_cache.get)
fold_loader = torch.utils.data.DataLoader(fold_ds, batch_size=config.BATCH_SIZE,
                                           shuffle=True, num_workers=0)

net_real = model.build_model().to(config.DEVICE)
optimizer_real = torch.optim.Adam(net_real.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY)

real_start = time.time()
n_batches = 0
for x_real, y_real in fold_loader:
    x_real, y_real = x_real.to(config.DEVICE), y_real.to(config.DEVICE)
    optimizer_real.zero_grad()
    with torch.autocast(device_type=device_type, enabled=config.USE_AMP):
        loss = loss_fn(net_real(x_real), y_real)
    scaler.scale(loss).backward()
    scaler.step(optimizer_real)
    scaler.update()
    n_batches += 1
torch.cuda.synchronize()
real_elapsed = time.time() - real_start

real_volumes_per_sec = len(fold_uids) / real_elapsed
print(f"\nreal-data throughput (warm cache, 1 epoch, {len(fold_uids)} volumes): "
      f"{real_volumes_per_sec:.1f} volumes/s over {real_elapsed:.1f}s "
      f"({n_batches} batches)")
print(f"GPU-only ceiling from the previous cell: {gpu_only_volumes_per_sec:.1f} volumes/s")
ratio = real_volumes_per_sec / gpu_only_volumes_per_sec
print(f"ratio: {ratio:.0%} of GPU-only throughput -- "
      f"{'data loading is NOT the bottleneck' if ratio > 0.7 else 'CPU-bound: data loading is slowing training down, worth investigating before running rung 4'}")

# Extrapolate: rung 3's own epoch counts ranged 12-30 per fold (25
# fold-runs total per rung-3-scale experiment). Rough estimate only --
# real epoch counts vary per experiment/seed, this just orders of
# magnitude before committing GPU time.
est_epoch_seconds = len(fold_uids) / real_volumes_per_sec
print(f"\nestimated seconds/epoch on a full-size fold (~{len(fold_uids)} volumes): {est_epoch_seconds:.1f}s")
for epochs in (12, 20, 30):
    total_minutes = est_epoch_seconds * epochs * 25 / 60  # 25 = 5 folds x 5 seed-repeats
    print(f"  if every fold runs ~{epochs} epochs: ~{total_minutes:.0f} min for one full "
          f"rung-3-scale experiment (25 fold-trainings)")

cache REUSED in 0.5s for 1362 volumes

real-data throughput (warm cache, 1 epoch, 1089 volumes): 742.8 volumes/s over 1.5s (137 batches)
GPU-only ceiling from the previous cell: 1055.8 volumes/s
ratio: 70% of GPU-only throughput -- data loading is NOT the bottleneck

estimated seconds/epoch on a full-size fold (~1089 volumes): 1.5s
  if every fold runs ~12 epochs: ~7 min for one full rung-3-scale experiment (25 fold-trainings)
  if every fold runs ~20 epochs: ~12 min for one full rung-3-scale experiment (25 fold-trainings)
  if every fold runs ~30 epochs: ~18 min for one full rung-3-scale experiment (25 fold-trainings)


**What we're looking for:** is training GPU-bound or CPU-bound (data
loading), and how many real minutes does one full rung-3-scale
experiment (25 fold-trainings) cost — so we can decide, before starting,
whether all 5 planned rung-4 experiments (family-oversampling, LR
schedule, augmentation, `class_weight`, denoising) fit in the time left
before 2026-09-16.

**What we found:** *(paste: CUDA device + memory; GPU-only synthetic
throughput + peak memory; cache REUSED/REBUILT + seconds; real-data
throughput + ratio vs. GPU-only; the three epoch-count time estimates)*

**Decision / next step:** *(if data loading isn't the bottleneck and the
time estimate fits comfortably: proceed to experiment 1 [family
oversampling] as planned. If CPU-bound, or the time estimate doesn't
fit: we need to cut scope — fewer experiments, fewer seed-repeats per
experiment, or a cheaper validation protocol than full nested CV —
before committing further GPU time.)*